#2. Add API Key Safely

In [72]:
from google.colab import userdata
from IPython.display import Markdown
api_key=userdata.get('main_api')

#4. First Simple Test

In [82]:
import requests
import json

# First API call with reasoning
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "qwen/qwen3.6-flash",
    "messages": [
        {
          "role": "user",
          "content": "How many r's are in the word 'strawberry'?"
        },


      ],
    "reasoning": {"enabled": True},
    'max_tokens':500,
  })
)


In [90]:
response_data=response.json()

In [95]:
Markdown(response_data['choices'][0]['message']['reasoning'])

Here's a thinking process:

1.  **Analyze User Input:**
   - Question: "How many r's are in the word 'strawberry'?"
   - Target word: "strawberry"
   - Task: Count the number of 'r' (case-insensitive typically, but here all lowercase) in the word.

2.  **Examine the Word:**
   - Write out the word: s-t-r-a-w-b-e-r-r-y
   - Identify each letter:
     1. s
     2. t
     3. r  (1)
     4. a
     5. w
     6. b
     7. e
     8. r  (2)
     9. r  (3)
     10. y

3.  **Count the 'r's:**
   - I see 'r' at position 3
   - I see 'r' at position 8
   - I see 'r' at position 9
   - Total count: 3

4.  **Verify:**
   - Word: strawberry
   - Breakdown: straw-berry
   - "straw" has 1 r
   - "berry" has 2 r's
   - Total: 1 + 2 = 3
   - Confirmed.

5.  **Formulate Response:**
   - State the answer clearly: There are 3 r's in the word "strawberry".
   - Keep it concise.
   - (Optional but helpful) Show the breakdown if needed, but direct answer is best.

6.  **Final Output Generation:**
   - "There are 3 r's in the word 'strawberry'."✅
   - Self-Correction/Verification during thought: This is a common trick question because people sometimes miscount due to how it's spoken, but the spelling is clear: s-t-R-a-w-b-e-R-R-y. It's definitely 3. I'll just give the direct, accurate answer. No extra fluff.✅
   - Output matches response.✅
   - Proceed. 
   - [Output Generation] -> "There are 3 r's in the word 'strawberry'."

#5. Simple AI Agent With Chat History

In [102]:
# Extract the assistant message with reasoning_details
#response = response.json()
#response = response['choices'][0]['message']

# Preserve the assistant message with reasoning_details
messages = [
  {"role": "user", "content": "How many r's are in the word 'strawberry'?"},
  {
    "role": "assistant",
    "content": response.get('content'),
    #"reasoning_details": response.get('reasoning_details')  # Pass back unmodified
  },
  {"role": "user", "content": "Are you sure? Think carefully."}
]

# Second API call - model continues reasoning from where it left off
response2 = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "qwen/qwen3.6-flash",
    "messages": messages,  # Includes preserved reasoning_details
    "reasoning": {"enabled": True},
    'max_tokens':500,

  })
)

#6. Test Agent

#7. Test Memory

#### First Prompt (The Input)
"Hey AI, remember this completely unique project statistic for my course: My students built an AI agent called Alpha-9 that successfully ran 4,350 automated code deployments across 12 different countries with a 99.4% success rate without a single human intervention."

#### Second Prompt (The Memory Test)
"What was the name of the agent my students built, and how many deployments did it run?"

In [105]:
import requests
import json

# Initialize chat history
chat_history = []

print("Simple AI Agent: Type 'quit' or 'exit' to end the conversation.")

while True:
    user_input = input("You: ")

    if user_input.lower() in ['quit', 'exit']:
        print("Agent: Goodbye!")
        break

    # Add user message to history
    chat_history = []
    chat_history.append({"role": "user", "content": user_input})

    try:
        # Make API call to OpenRouter with the current chat history
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
            },
            data=json.dumps({
                "model": "qwen/qwen3.6-flash", # Using the model from previous successful calls
                "messages": chat_history,
                "max_tokens": 500, # Keep max_tokens to avoid credit issues
                "reasoning": {"enabled": True}
            })
        )

        if response.ok:
            response_data = response.json()
            try:
                assistant_message_content = response_data['choices'][0]['message']['content']
                print(f"Agent: {assistant_message_content}")
                # Add assistant's response to history
                chat_history.append({"role": "assistant", "content": assistant_message_content})
            except KeyError as e:
                print(f"Agent Error: Expected key not found in response: {e}")
                print(f"Full response data: {response_data}")
        else:
            print(f"Agent Error: API call failed with status code {response.status_code}:")
            print(response.text)
            # If API call fails, do not add to history to avoid propagating bad states
            chat_history.pop() # Remove the last user message to try again

    except requests.exceptions.RequestException as e:
        print(f"Agent Error: Network or request issue: {e}")
        chat_history.pop() # Remove the last user message to try again


Simple AI Agent: Type 'quit' or 'exit' to end the conversation.
You: my name is yisak and from addis ababa
Agent: Hi Yisak! Nice to meet you. 👋 Addis Ababa is a wonderful city with a rich culture, great food, and such a vibrant atmosphere. How can I help you today? Whether you have a question, need advice, or just want to chat, I'm here for you! 😊
You: what is my name and where am i from
Agent: I don't actually know your name or where you're from. For privacy and security reasons, I don't have access to personal information, account details, or memory of past conversations. If you'd like to share your name or location, I'd be happy to chat with you using that! Otherwise, what can I help you with today?
You: quit 
Agent: Got it. I'm ending this session now. Feel free to reach out anytime if you need assistance in the future. Take care! 👋
You: quit
Agent: Goodbye!


In [107]:
import requests
import json

# 1. Define real local Python tools that the agent can execute
def calculate(expression: str) -> str:
    """Safely evaluate simple arithmetic expressions."""
    try:
        # Using a restricted global dictionary for basic math safety
        allowed_chars = "0123456789+-*/(). "
        if not all(char in allowed_chars for char in expression):
            return "Error: Invalid characters in mathematical expression."
        return str(eval(expression, {"__builtins__": None}, {}))
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"

def get_weather(location: str) -> str:
    """Mock weather tool that returns data based on the city."""
    city = location.lower()
    if "london" in city:
        return "15°C and raining heavily."
    elif "tokyo" in city:
        return "22°C and clear skies."
    elif "addis ababa" in city:
        return "24°C and beautifully sunny."
    else:
        return "18°C and partly cloudy."

# Mapping tool names to their actual Python functions
AVAILABLE_TOOLS = {
    "calculate": calculate,
    "get_weather":get_weather
}

# 2. Define the JSON schemas so the LLM knows what tools are available
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Use this tool to solve math equations or complex arithmetic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The math expression to solve, e.g., '2350 * 1.15'"
                    }
                },
                "required": ["expression"]
            }
        },
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a specific city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city name, e.g., 'London' or 'Tokyo'"
                    }
                },
                "required": ["location"]
            }
        }

    }
]

# Initialize chat history with a system prompt setting agent rules
chat_history = [
    {"role": "system", "content": "You are an autonomous AI Agent equipped with local tools. Use your tools whenever you need to resolve complex logic or math questions precisely."}
]

print("Agentic AI System Active. Type 'quit' or 'exit' to end.")

while True:
    user_input = input("\nYou: ")

    if user_input.lower() in ['quit', 'exit']:
        print("Agent: Goodbye!")
        break

    # Add user message to history
    chat_history.append({"role": "user", "content": user_input})

    # The Agent Reasoning Loop
    # We loop up to 5 times to let the agent call multiple tools back-to-back if needed
    max_loops = 5
    for loop_count in range(max_loops):
        try:
            response = requests.post(
                url="https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {api_key}",
                    "Content-Type": "application/json",
                },
                data=json.dumps({
                    "model": "qwen/qwen3.6-flash",
                    "messages": chat_history,
                    "tools": tools_schema,       # Pass available tools to the LLM
                    "tool_choice": "auto",       # Let the LLM choose if it needs a tool
                    "max_tokens": 500
                })
            )

            if not response.ok:
                print(f"API Error ({response.status_code}): {response.text}")
                chat_history.pop()  # Safe fallback rollback
                break

            response_data = response.json()
            message_data = response_data['choices'][0]['message']

            # CRITICAL AGENTIC CHECK: Did the LLM request to run a tool?
            if 'tool_calls' in message_data and message_data['tool_calls']:
                # Append the assistant's decision to call the tool into history
                chat_history.append(message_data)

                for tool_call in message_data['tool_calls']:
                    tool_name = tool_call['function']['name']
                    tool_args = json.loads(tool_call['function']['arguments'])
                    tool_id = tool_call['id']

                    print(f"\n[Agent Thought]: I need to use the '{tool_name}' tool with arguments: {tool_args}")

                    # Run the local function
                    if tool_name in AVAILABLE_TOOLS:
                        tool_function = AVAILABLE_TOOLS[tool_name]
                        # Extract parameter values dynamically
                        execution_result = tool_function(**tool_args)
                    else:
                        execution_result = f"Error: Tool '{tool_name}' is not registered locally."

                    print(f"[Tool Output]: {execution_result}")

                    # Send the result of the tool run back to the AI's history
                    chat_history.append({
                        "role": "tool",
                        "tool_call_id": tool_id,
                        "name": tool_name,
                        "content": execution_result
                    })

                # Continue the loop so the model can read the tool's results and answer
                continue

            else:
                # No tools were requested, this is the final verbal response
                final_answer = message_data['content']
                print(f"\nAgent: {final_answer}")

                # Fixes original code memory bug: save the answer so it remembers next turn!
                chat_history.append({"role": "assistant", "content": final_answer})
                break # Exit the reasoning loop since the problem is resolved

        except requests.exceptions.RequestException as e:
            print(f"Network Error: {e}")
            chat_history.pop()
            break

Agentic AI System Active. Type 'quit' or 'exit' to end.

You: What is the weather like in Addis Ababa right now?

[Agent Thought]: I need to use the 'get_weather' tool with arguments: {'location': 'Addis Ababa'}
[Tool Output]: 24°C and beautifully sunny.

Agent: The weather in Addis Ababa right now is 24°C and beautifully sunny.

You: What is the weather like in Accar right now?

[Agent Thought]: I need to use the 'get_weather' tool with arguments: {'location': 'Accar'}
[Tool Output]: 18°C and partly cloudy.

Agent: The weather in Accar is currently 18°C and partly cloudy.

You: quit
Agent: Goodbye!


1. What is 4350 times 12?
2. What is the weather like in Addis Ababa right now?

